In [ ]:
import os, gc

import numpy as np
import pickle as pkl

from numba import njit, prange, float64, int64
from scipy.spatial import cKDTree
from meshpy.tet import MeshInfo, build, Options

from illustris_python.snapshot import loadSubset as illustris_load

---
---
---

In [ ]:
def mean_edge_distances(core_coords, core_min, core_max, frac=0.1):

    '''
    frac - fraction of the grid size that we take as an edge
    '''
    
    
    edges = []
    L = core_max-core_min

    for i in range(3):
        for sign in [0,1]:

            # define the slice
            if sign == 0:
                mask = (core_coords[:, i] >= core_min[i]) & (core_coords[:, i] <= core_min[i] + frac*L[i])
                region_min    = core_min.copy()
                region_max    = core_max.copy()
                region_max[i] = core_min[i] + frac*L[i]
            else:
                mask = (core_coords[:, i] <= core_max[i]) & (core_coords[:, i] >= core_max[i] - frac*L[i])
                region_min    = core_min.copy()
                region_max    = core_max.copy()
                region_min[i] = core_max[i] - frac*L[i]

            # select particles
            pts = core_coords[mask]
            N = len(pts)
            if N == 0:
                mean_dist = np.max(core_max-core_min)
            else:
                V = np.prod(region_max-region_min)
                mean_dist = (V/N)**(1/3)

            edges.append(mean_dist)

    return edges

---
---
---

In [ ]:
def compute_tetgen_delaunay(coords):

    '''
    mesh.points   - (vertex coordinates) The actual coordinates of all vertices (the same as the input
                        coords array if we use Options("Q"), larger if we use Options("q")).
    mesh.elements - (tetrahedra) This is the connectivity. It lists which 4 points define each tetrahedron.
    '''
    
    mesh_info = MeshInfo()         # object to hold your input geometry (the particle positions)
    mesh_info.set_points(coords)   # loads the N particle positions into tetgen

    # Q - no refinement flags (i.e. quality constraints), so runs a straight Delaunay tetrahedralization
    # q - refine or insert extra "Steiner points" (e.g. for constrained Delaunay, surface meshing, or quality improvement)
    opts = Options("Q")
    mesh = build(mesh_info, options=opts)
    
    return np.array(mesh.elements, dtype=np.int64)

---

**Tetrahedron volume:**

$$
V_{\text{tetra}}
=
\frac{1}{6} \left| \det \left(
\begin{bmatrix}
\mathbf{v}_1 & \mathbf{v}_2 & \mathbf{v}_3
\end{bmatrix} \right) \right|
,
$$
where
$(\mathbf{v}_1 = \mathbf{p}_1 - \mathbf{p}_0)$,
$(\mathbf{v}_2 = \mathbf{p}_2 - \mathbf{p}_0)$,
$(\mathbf{v}_3 = \mathbf{p}_3 - \mathbf{p}_0)$.

---

**Vertex density assignment in your code:**

$$
\rho_i
=
\frac{m}{\frac{1}{4} \sum\limits_{t \ni i} V_t}
=
\frac{m}{V_{contign}(i)}
,
$$
where the sum runs over all tetrahedra $t$ that contain vertex $i$ and the $\frac{1}{4}$ factor appears to average the four tetrahedra around vertex $i$: this is known as the contiguous Voronoi cell volume.

In [ ]:
@njit(inline='always')
def tetra_volume(coords, connection):

    '''
    Tetrahedron volume formula.
    '''
    
    p0 = coords[connection[0]]
    v1 = coords[connection[1]] - p0
    v2 = coords[connection[2]] - p0
    v3 = coords[connection[3]] - p0

    # same as abs(np.linalg.det(np.stack((v1, v2, v3))))/6.0
    # but like 3x faster
    return abs(v1[0]*(v2[1]*v3[2] - v2[2]*v3[1]) +
               v1[1]*(v2[2]*v3[0] - v2[0]*v3[2]) +
               v1[2]*(v2[0]*v3[1] - v2[1]*v3[0])) / 6.0

In [ ]:
@njit
def compute_densities(coords, connections, m=1):
    
    rho = np.zeros(len(coords), dtype='float64')
    
    for connection in connections:
        # the tetrahedron's volume
        vol = tetra_volume(coords, connection)
        # for each vertex we add the volume
        for i0 in connection: rho[i0] += vol

    # 4 fro the number of vertices in the tetrahedron
    # We do not use the mass normalization, because we chose
    #     at this stage to compute the number density, not
    #     the mass one.
    return 4*m/rho

---

In [ ]:
@njit(inline='always')
def compute_gradients(coords, connections, rho):
    
    Drho = np.empty((len(connections), 3), dtype=np.float64)

    for i, (c0, c1, c2, c3) in enumerate(connections):
        p0 = coords[c0]; p1 = coords[c1]; p2 = coords[c2]; p3 = coords[c3]
        r0 = rho[c0];    r1 = rho[c1];    r2 = rho[c2];    r3 = rho[c3]
        
        E = np.empty((3, 3))
        E[:,0] = p1 - p0
        E[:,1] = p2 - p0
        E[:,2] = p3 - p0
        
        dltrho = np.array([r1 - r0, r2 - r0, r3 - r0])
        Drho[i] = np.linalg.solve(E.T, dltrho)
        
    return Drho

---

In [ ]:
@njit(parallel=True, fastmath=True)
def precompute_inverses_fast(tetra_points):
    
    n = tetra_points.shape[0]
    Ainv    = np.empty((n, 3, 3), dtype=np.float64)
    origins = np.empty((n, 3),    dtype=np.float64)

    for i in prange(n):
        p0 = tetra_points[i, 0]
        T  = tetra_points[i, 1:] - p0
        a,b,c = T[0]; d,e,f = T[1]; g,h,k = T[2]

        det = a*(e*k - f*h) - b*(d*k - f*g) + c*(d*h - e*g)
        inv_det = 1.0 / det

        # inv(T): (row-wise cofactors)/det
        r00 =  (e*k - f*h) * inv_det
        r01 = -(b*k - c*h) * inv_det
        r02 =  (b*f - c*e) * inv_det
        r10 = -(d*k - f*g) * inv_det
        r11 =  (a*k - c*g) * inv_det
        r12 = -(a*f - c*d) * inv_det
        r20 =  (d*h - e*g) * inv_det
        r21 = -(a*h - b*g) * inv_det
        r22 =  (a*e - b*d) * inv_det

        # inv(E) = inv((T)^T) = (inv(T))^T
        Ainv[i,0,0] = r00; Ainv[i,0,1] = r10; Ainv[i,0,2] = r20
        Ainv[i,1,0] = r01; Ainv[i,1,1] = r11; Ainv[i,1,2] = r21
        Ainv[i,2,0] = r02; Ainv[i,2,1] = r12; Ainv[i,2,2] = r22

        origins[i] = p0

    return Ainv, origins

---

In [ ]:
@njit(parallel=True, fastmath=True)
def compute_density_grid(coords_flat, candidates, origins, Ainv_all, coords, connections, rho, eps=1e-10, near_tau=5e-2):

    zero_points = 0
    n_points = coords_flat.shape[0]
    density  = np.empty(n_points, dtype=np.float64)

    for i in prange(n_points):
        p = coords_flat[i]
        found = False
        best_idx = -1
        best_min = -1e300

        for j in range(candidates.shape[1]):
            t = candidates[i, j]
            v = p - origins[t]
            b = Ainv_all[t] @ v
            w = 1.0 - (b[0] + b[1] + b[2])

            # the most "inside-ish" candidate
            m = b[0]
            if b[1] < m: m = b[1]
            if b[2] < m: m = b[2]
            if w    < m: m = w
            if m > best_min:
                best_min = m
                best_idx = t

            if (b[0] >= -eps) and (b[1] >= -eps) and (b[2] >= -eps) and (w >= -eps):
                a,bv,cv,dv = connections[t,0], connections[t,1], connections[t,2], connections[t,3]
                
                b0 = b[0] if b[0] > 0.0 else 0.0
                b1 = b[1] if b[1] > 0.0 else 0.0
                b2 = b[2] if b[2] > 0.0 else 0.0
                ww = w     if w     > 0.0 else 0.0
                s  = ww + b0 + b1 + b2
                if s != 1.0:
                    invs = 1.0 / s
                    ww *= invs; b0 *= invs; b1 *= invs; b2 *= invs

                density[i] = ww*rho[a] + b0*rho[bv] + b1*rho[cv] + b2*rho[dv]
                found = True; break

        if found: continue

        zero_points += 1

        # Fallback with a two-stage, local estimate:
        # After testing the k candidate tets with a tolerant inside-check, if none contain the point we pick the “best near-miss” tetra
        #     (the one whose smallest barycentric weight is largest), clamp any tiny negative barycentric weights to zero, renormalize 
        #     them to sum to 1, and interpolate its vertex densities—yielding a smooth, nonnegative value.
        # If even that’s not usable, we fall back to the nearest vertex’s density of that best tetra. This prevents artificial holes, 
        #     preserves continuity, and plays nicely with the median/open/close filters (in the next notebook).

        # clamp & renormalize if not too far outside
        if best_idx != -1 and best_min > -near_tau:
            t = best_idx
            v = p - origins[t]
            b = Ainv_all[t] @ v
            w = 1.0 - (b[0] + b[1] + b[2])

            if b[0] < 0.0: b[0] = 0.0
            if b[1] < 0.0: b[1] = 0.0
            if b[2] < 0.0: b[2] = 0.0
            if w     < 0.0: w  = 0.0
            s = w + b[0] + b[1] + b[2]

            if s > 0.0:
                invs = 1.0 / s
                w *= invs; b[0] *= invs; b[1] *= invs; b[2] *= invs
                a,bv,cv,dv = connections[t,0], connections[t,1], connections[t,2], connections[t,3]
                density[i] = w*rho[a] + b[0]*rho[bv] + b[1]*rho[cv] + b[2]*rho[dv]
                continue

        # Last resort: piecewise-constant from nearest vertex of best tetrahedron
        if best_idx != -1:
            t = best_idx
            c0,c1,c2,c3 = connections[t,0], connections[t,1], connections[t,2], connections[t,3]
            
            d0 = (p[0]-coords[c0,0])**2 + (p[1]-coords[c0,1])**2 + (p[2]-coords[c0,2])**2
            d1 = (p[0]-coords[c1,0])**2 + (p[1]-coords[c1,1])**2 + (p[2]-coords[c1,2])**2
            d2 = (p[0]-coords[c2,0])**2 + (p[1]-coords[c2,1])**2 + (p[2]-coords[c2,2])**2
            d3 = (p[0]-coords[c3,0])**2 + (p[1]-coords[c3,1])**2 + (p[2]-coords[c3,2])**2
            
            rb = c0; rd = d0
            
            if d1 < rd: rb, rd = c1, d1
            if d2 < rd: rb, rd = c2, d2
            if d3 < rd: rb, rd = c3, d3
            density[i] = rho[rb]
        
        else:
            density[i] = 0.0

    return density, zero_points

---
---
---